# TabICLv2 Classifier artifact inference — standalone Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-classifier-pipeline/blob/main/tutorials/tabiclv2_classifier_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL)
[![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl)
[![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

**Profile:** `ARTIFACT-INFERENCE` · **DIMER Notebook Specification:** `1.0`

Someone hands you a TabICL serving bundle, produced by the main tutorial or by the DIMER pipeline, and asks for predictions on new rows. Before you load it you want three things: that the archive is internally consistent (and, when you were given its SHA-256 by a trusted producer, that it is the archive they sent), that it carries everything an in-context model needs (checkpoint **and** training context), and that no code runs beyond the pinned `tabicl` estimator.

This notebook performs **no gradient fine-tuning**. TabICL's required `fit(context_X, context_y)` call only registers the in-context support table before prediction.

**By the end of this notebook you will be able to:**
- **Verify** a bundle: safe extraction, manifest format and version, per-file SHA-256 of the checkpoint and the training context (internal consistency), and, only when you supply a trusted whole-ZIP SHA-256, integrity of the archive itself.
- **Reconstruct** the classifier from the bundle alone, with the inference settings the producer recorded.
- **Score** a new CSV, with the producer's categorical encoders applied identically, and download `predictions.csv`.

**This notebook does not demonstrate:** gradient fine-tuning, model selection, or creation of the artifact it consumes. The artifact must be supplied from outside this notebook execution.

> **Trust boundary:** load only artifacts you created yourself or obtained from a trusted source. Safe ZIP extraction prevents path traversal; it does not make a PyTorch checkpoint trustworthy. Paste a known ZIP SHA-256 below when available.

## Prerequisites
- A bundle ZIP (`tabiclv2-classifier-artifact.zip` from the main tutorial, or a DIMER serving bundle in the `tabicl-dimer-classifier-v1` format) and, ideally, the SHA-256 printed at export.
- A CSV of new rows with the bundle's feature columns (no label needed).
- **Runtime:** CPU is sufficient; when CUDA is available the estimator may use it automatically. About two minutes end to end on the small verification batch.


## 1. Install and verify the matching runtime

The notebook installs from the same fully resolved Python 3.12 release lock graph as the producer tutorial. In a cloned repository it uses `tutorials/requirements-release.lock`; standalone Colab fetches the lock from GitHub and verifies its embedded SHA-256 before installation. The artifact's recorded TabICL version is checked against that locked runtime before deserialization, and the effective Python, TabICL, PyTorch, and accelerator identities are printed.


In [ ]:
import hashlib
import importlib.metadata
import subprocess
import sys
import urllib.request
from pathlib import Path

RELEASE_LOCK_SHA256 = "64e9a167567495263694f555195ca7df4488016cc76ab1e327e480509a4ac6d5"
RELEASE_LOCK_URL = "https://raw.githubusercontent.com/kurtvalcorza/tabicl-classifier-pipeline/main/tutorials/requirements-release.lock"
LOCAL_RELEASE_LOCK = Path("tutorials/requirements-release.lock")
RUNTIME_RELEASE_LOCK = Path("/tmp/tabicl-classifier-requirements-release.lock")

if LOCAL_RELEASE_LOCK.exists():
    lock_payload = LOCAL_RELEASE_LOCK.read_bytes()
    lock_source = str(LOCAL_RELEASE_LOCK)
else:
    with urllib.request.urlopen(RELEASE_LOCK_URL, timeout=30) as response:
        lock_payload = response.read()
    lock_source = RELEASE_LOCK_URL

observed_lock_sha256 = hashlib.sha256(lock_payload).hexdigest()
if observed_lock_sha256 != RELEASE_LOCK_SHA256:
    raise RuntimeError(
        f"Release dependency lock SHA-256 mismatch: {observed_lock_sha256}; "
        f"expected {RELEASE_LOCK_SHA256}"
    )
RUNTIME_RELEASE_LOCK.write_bytes(lock_payload)
print("Release dependency lock:", lock_source)
print("Release dependency lock SHA-256:", observed_lock_sha256)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(RUNTIME_RELEASE_LOCK)],
    check=True,
)

import torch

TABICL_VERSION = "2.1.1"
if importlib.metadata.version("tabicl") != TABICL_VERSION:
    raise RuntimeError("Unexpected tabicl version")
print("Python:", sys.version.split()[0])
for package in ['tabicl', 'pyarrow', 'pandas', 'numpy', 'scikit-learn', 'torch']:
    print(f"{package}:", importlib.metadata.version(package))
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 2. Upload and verify the artifact ZIP

In order: the archive's SHA-256 is computed and, if you pasted `EXPECTED_ZIP_SHA256`, compared (**this is the only check that establishes the archive is the one your producer sent**; without it, a bundle whose files and manifest were altered together still passes); every member is checked for absolute paths, `..` segments and symlinks and for escaping the extraction folder; the archive is extracted; exactly one `artifact.json` must exist; its `artifactFormat` and `tabiclVersion` must match this notebook; the checkpoint and training-context paths named in the manifest are resolved *inside* the bundle root; and each file's SHA-256 must equal the digest recorded in the manifest.

**What to look for:** `✓ Artifact structure and digests verified`, which means the bundle is well-formed and internally consistent. Any other outcome means stop and ask the producer.


**Data handling / privacy.** Selecting the artifact sends its bytes—including the persisted labelled training context—to the active Google Colab runtime. The notebook does not send the artifact, its embedded rows, or later inference rows to an external inference API or to Hugging Face; model auto-download is disabled. Package installation still contacts the configured Python package index. Do not upload confidential, restricted, sensitive, or regulated artifacts unless you are authorized to process them in the selected runtime.


In [ ]:
import csv
import hashlib
import io
import json
import shutil
import stat
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

EXPECTED_ZIP_SHA256 = ""  # @param {type:"string"}
MAX_ARCHIVE_MEMBERS = 1000
MAX_ARCHIVE_EXPANDED_BYTES = 2 * 1024**3


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract_zip(zip_path, dest):
    """Extract only after every member passed the path and symlink checks."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        infos = archive.infolist()
        if len(infos) > MAX_ARCHIVE_MEMBERS:
            raise ValueError(f"Artifact has too many members: {len(infos)}")
        expanded_bytes = sum(info.file_size for info in infos if not info.is_dir())
        if expanded_bytes > MAX_ARCHIVE_EXPANDED_BYTES:
            raise ValueError(f"Artifact expands to {expanded_bytes} bytes, above the {MAX_ARCHIVE_EXPANDED_BYTES}-byte limit")
        for info in infos:
            if "\\" in info.filename:
                raise ValueError(f"Ambiguous backslash path in ZIP member: {info.filename}")
            name = info.filename
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            target = (dest / Path(name)).resolve()
            if root != target and root not in target.parents:
                raise ValueError("ZIP member escapes destination")
        archive.extractall(dest)


def manifest_member_path(root, value, field):
    """Resolve a manifest path inside the bundle root, refusing absolute paths and traversal."""
    raw_value = str(value)
    if "\\" in raw_value:
        raise ValueError(f"Ambiguous backslash {field} path in artifact.json: {value!r}")
    rel = Path(raw_value)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")
    root_resolved = Path(root).resolve()
    target = (root_resolved / rel).resolve()
    if root_resolved != target and root_resolved not in target.parents:
        raise ValueError(f"{field} path escapes artifact root: {value!r}")
    return target


uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one artifact ZIP")
name, payload = next(iter(uploaded.items()))
zip_path = Path("/content") / Path(name).name
zip_path.write_bytes(payload)
observed = sha256_file(zip_path)
print("ZIP SHA-256:", observed)
if EXPECTED_ZIP_SHA256:
    expected = EXPECTED_ZIP_SHA256.strip().lower()
    if len(expected) != 64 or any(character not in "0123456789abcdef" for character in expected):
        raise ValueError("EXPECTED_ZIP_SHA256 must be 64 hex chars")
    if observed != expected:
        raise RuntimeError("Artifact ZIP SHA-256 mismatch")

extract_dir = Path("/content/tabiclv2-classifier-artifact")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
safe_extract_zip(zip_path, extract_dir)

matches = list(extract_dir.rglob("artifact.json"))
if len(matches) != 1:
    raise ValueError("Expected exactly one artifact.json")
root = matches[0].parent
manifest = json.loads(matches[0].read_text())
if manifest.get("artifactFormat") != "tabicl-dimer-classifier-v1":
    raise ValueError(f"Unsupported artifactFormat: {manifest.get('artifactFormat')}")
if manifest.get("tabiclVersion") != TABICL_VERSION:
    raise ValueError("Artifact TabICL version does not match notebook pin")

ckpt = manifest_member_path(root, manifest["checkpoint"], "checkpoint")
context_path = manifest_member_path(root, manifest["trainingContext"], "trainingContext")
if sha256_file(ckpt) != manifest["digests"]["checkpointSha256"]:
    raise RuntimeError("Checkpoint digest mismatch")
if sha256_file(context_path) != manifest["digests"]["trainingContextSha256"]:
    raise RuntimeError("Training-context digest mismatch")

sizes = manifest.get("sizes")
if sizes is not None:
    if ckpt.stat().st_size != sizes.get("checkpointBytes"):
        raise RuntimeError("Checkpoint size mismatch")
    if context_path.stat().st_size != sizes.get("trainingContextBytes"):
        raise RuntimeError("Training-context size mismatch")
else:
    print("⚠ Legacy DIMER v1 manifest has no checkpointBytes/trainingContextBytes; global archive limits and SHA-256 checks still apply.")

root_prefix = root.relative_to(extract_dir)
expected_files = {
    (root_prefix / "artifact.json").as_posix(),
    (root_prefix / Path(manifest["checkpoint"])).as_posix(),
    (root_prefix / Path(manifest["trainingContext"])).as_posix(),
}
observed_files = {path.relative_to(extract_dir).as_posix() for path in extract_dir.rglob("*") if path.is_file()}
unexpected_files = sorted(observed_files - expected_files)
if unexpected_files:
    print("⚠ Unexpected unlisted artifact file(s) retained for DIMER v1 compatibility:", unexpected_files)
print("✓ Artifact structure and digests verified")
print("Artifact provenance and runtime contract:")
print("  artifactFormat:", manifest.get("artifactFormat"))
print("  tabiclVersion:", manifest.get("tabiclVersion"))
print("  baseCheckpoint:", manifest.get("baseCheckpoint"))
print("  baseModelRevision:", manifest.get("baseModelRevision"))
print("  baseModelSha256:", manifest.get("baseModelSha256"))
effective_base_model_source = manifest.get("baseModelSource")
if effective_base_model_source is None:
    effective_base_model_source = manifest.get("checkpointSource")
print("  baseModelSource:", effective_base_model_source)
print("  baseMatchesPinned:", manifest.get("baseMatchesPinned"))
if manifest.get("checkpointSource") is not None:
    print("  checkpointSource (legacy/tutorial):", manifest.get("checkpointSource"))
print("  checkpointSha256:", manifest.get("digests", {}).get("checkpointSha256"))
print("  wholeArchiveSha256:", observed)
print("  mode:", manifest.get("mode"))
print("  selectionBasis:", manifest.get("selectionBasis"))
print("  runtimeTabICL:", TABICL_VERSION)
print("  runtimePyTorch:", torch.__version__)


## 3. Reconstruct the in-context classifier

The bundle's `inference` block records how the producer ran the model: ensemble size, random seed, many-class support, and the categorical encoders fitted on the training split. The classifier is built from the bundled checkpoint with `allow_auto_download=False` (nothing may be fetched behind your back) and then given the training context with `fit`, which registers those rows as its prompt and trains nothing.


The producer's recorded `randomState` is restored. The remaining run-to-run variability can come from the effective PyTorch/CUDA build, hardware-specific numerical kernels, and runtime scheduling; the seed does not imply bitwise identity across machines. No gradient optimization occurs in this notebook.


In [ ]:
from tabicl import TabICLClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
context = pd.read_parquet(context_path)
FEATURE_COLUMNS = manifest["featureColumns"]
TARGET_COLUMN = manifest["targetColumn"]
inference = manifest["inference"]
model = TabICLClassifier(
    model_path=str(ckpt), allow_auto_download=False,
    n_estimators=inference["nEstimators"], random_state=inference["randomState"],
    device=DEVICE, support_many_classes=inference.get("supportManyClasses", True))
model.fit(context[FEATURE_COLUMNS], context[TARGET_COLUMN])
print(f"✓ Loaded with {len(context)} context rows and {len(FEATURE_COLUMNS)} features; classes: {list(model.classes_)}")
print("Effective inference device:", DEVICE)
print("Expected inference CSV feature columns:", FEATURE_COLUMNS)


## 4. Upload rows and predict

Upload one CSV with the bundle's feature columns. Order does not matter and extra columns are preserved in the output; duplicate header names, missing features, or pre-existing `prediction` / `probability_*` columns stop the run. Categoricals are encoded with the producer's map, and values that map to "unknown" are counted in a warning, because a batch full of unseen categories is usually a schema drift you want to know about.

The output adds `prediction` (the most probable class) and one `probability_<class>` column per class. For cost-sensitive decisions, threshold the relevant probability rather than using the argmax.


Before upload, use the exact feature list printed by Step 3. The inference CSV does not need the target column. Uploading the CSV sends it to the active Google Colab runtime; prediction is performed locally in that runtime with `allow_auto_download=False`, and the only outbound action after scoring is the explicit download of `predictions.csv` back to your browser.


In [ ]:
def raw_header(payload):
    """First non-empty CSV row, read before pandas can rename duplicate names."""
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(cell.strip() for cell in row):
            return row
    raise ValueError("CSV has no header")


def read_inference_csv(payload, feature_columns):
    header = raw_header(payload)
    seen, dupes = set(), []
    for column in header:
        if column in seen and column not in dupes:
            dupes.append(column)
        seen.add(column)
    if dupes:
        raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")
    frame = pd.read_csv(io.BytesIO(payload))
    if "prediction" in frame.columns or any(column.startswith("probability_") for column in frame.columns):
        raise ValueError("Inference CSV already contains prediction/probability output columns")
    missing = [column for column in feature_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV missing features: {missing}")
    return frame


def apply_encoder(frame, encoders, label="data"):
    """Apply the producer's ordinal maps; unseen or missing values get the extra 'unknown' code."""
    out = frame.copy()
    for column, categories in encoders.items():
        lookup = {category: index for index, category in enumerate(categories)}
        unknown = len(categories)
        encoded, unseen = [], 0
        for value in out[column]:
            if pd.isna(value):
                encoded.append(unknown)
                continue
            key = str(value)
            if key not in lookup:
                unseen += 1
            encoded.append(lookup.get(key, unknown))
        out[column] = encoded
        if unseen:
            print(f"⚠ {label}: {unseen} unseen categorical value(s) in {column!r} encoded as unknown.")
    return out


uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one inference CSV")
_, payload = next(iter(uploaded.items()))
rows = read_inference_csv(payload, FEATURE_COLUMNS)
X = apply_encoder(rows[FEATURE_COLUMNS], inference.get("categoricalEncoders", {}), "inference CSV")
pred = np.asarray(model.predict(X))
proba = np.asarray(model.predict_proba(X))
out = rows.copy()
out["prediction"] = pred
for index, class_label in enumerate(model.classes_):
    out[f"probability_{class_label}"] = proba[:, index]
output_path = Path("/content/tabiclv2_classifier_predictions.csv")
out.to_csv(output_path, index=False)
print(f"✓ Wrote {len(out)} predictions to {output_path}")
files.download(str(output_path))


## When a check fails

| Error | Meaning | What to do |
|---|---|---|
| `Artifact ZIP SHA-256 mismatch` | not the archive whose digest you were given | get it again; never edit the expected digest |
| `Unsafe ZIP member` / `ZIP member escapes destination` | the archive tries to write outside its folder | reject it; that is what a malicious archive looks like |
| `Unsupported artifactFormat` | a regressor bundle, or not a TabICL bundle | use the matching notebook |
| `Artifact TabICL version does not match notebook pin` | produced with another `tabicl` release | install that release, or re-export |
| `Checkpoint digest mismatch` / `Training-context digest mismatch` | a file inside the bundle differs from its manifest entry | reject the bundle |
| `Inference CSV missing features` | schema mismatch | add the listed columns with the training names |
| many `unseen categorical value(s)` | the new data uses categories the producer never saw | check for a schema change before trusting the predictions |

## AI provenance

This inference tutorial was developed with substantial AI assistance under maintainer direction and review: original build by **GPT-5.6 Sol High**, via **OpenAI / ChatGPT**, under Agent Relay role **Builder**; content revision by **Claude Fable 5.1**, via **Anthropic / Claude Code**, under Agent Relay role **Reviewer and Builder**; review refinement and companion discoverability by **Gemini 3.8 Flash High**, via **Google DeepMind / Antigravity**, under Agent Relay role **Builder**. Provenance only; not independent sign-off.


## Interpretation and limits

A successful run establishes that the externally supplied archive passed the documented path/member/expanded-size checks, that its manifest format and TabICL version are compatible, that the checkpoint and training-context digests match the manifest, that any recorded component sizes match, that the serving state reconstructs without network model fallback, and that a schema-compatible new CSV can be scored and exported. If you supplied an out-of-band `EXPECTED_ZIP_SHA256`, success also establishes that the whole archive matches that producer-provided digest.

It **does not prove** that an artifact is trustworthy when no authentic out-of-band digest or trusted producer relationship exists; internal consistency can be forged together. It also does not establish calibration, fairness, production fitness, or distributional compatibility of the new rows—schema-valid data can still be out of distribution. Treat large unseen-category warnings as drift signals and validate decision thresholds and task-specific costs on appropriate held-out data before consequential use.

**Next:** reproduce known holdout predictions from the producer, test representative new batches, investigate schema/drift warnings, and validate calibration/threshold behavior for the intended deployment population.
